In [0]:
%run "/Users/manasadevi.j@diggibyte.com/DATABRICKS-ASSIGNMENTS/src/Question-1/source_to_bronze/utils"

In [0]:


from pyspark.sql.types import *
from pyspark.sql.functions import current_date
import re

# 2. Schemas
employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("EmployeeName", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Age", IntegerType(), True)
])

department_schema = StructType([
    StructField("DepartmentID", StringType(), True),
    StructField("DepartmentName", StringType(), True)
])

country_schema = StructType([
    StructField("CountryCode", StringType(), True),
    StructField("CountryName", StringType(), True)
])

# 3. Read Bronze (Volumes)
bronze_path = "/Volumes/workspace/assignments/dev_catalog/bronze"

employee_df = spark.read.format("csv").option("header", True).schema(employee_schema)\
    .load(f"{bronze_path}/employee")

department_df = spark.read.format("csv").option("header", True).schema(department_schema)\
    .load(f"{bronze_path}/department")

country_df = spark.read.format("csv").option("header", True).schema(country_schema)\
    .load(f"{bronze_path}/country")

# 4. CamelCase → snake_case
def camel_to_snake(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def convert_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, camel_to_snake(c))
    return df

employee_df = convert_columns(employee_df)
department_df = convert_columns(department_df)
country_df = convert_columns(country_df)

# 5. Add load_date
employee_df = employee_df.withColumn("load_date", current_date())
department_df = department_df.withColumn("load_date", current_date())
country_df = country_df.withColumn("load_date", current_date())

# 6. Write Silver 
silver_path = "/Volumes/workspace/assignments/dev_catalog/silver"

employee_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{silver_path}/dim_employee")

department_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{silver_path}/dim_department")

country_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{silver_path}/dim_country")

print("✅ Silver Layer Created Successfully (Schema Issue Fixed)")